In [ ]:
import scanpy as sc
import pandas as pd
import os

In [ ]:
#repeat the following for each dataset in both species to make .h5ad files
Nvec_mtx_directory = "./Nvec_cnido_matrix_files/"
output_h5ad_file = "./Nvec_cnido_matrix_files/Nvec_cnido.h5ad"

In [ ]:
path = os.path.join(Nvec_mtx_directory, "features.tsv")

df = pd.read_csv(path, header=None)

# split into two parts
split = df[0].str.split(" ", n=1, expand=True)

gene_id = split[0]
gene_symbol = split[1].str.replace('"', '', regex=False)

features_fixed = pd.DataFrame({
    0: gene_id,
    1: gene_symbol,
    2: "Gene Expression"
})

features_fixed.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "features.tsv")

# read tsv skipping first row
df = pd.read_csv(path, sep="\t", header=None, skiprows=1)

# overwrite original file
df.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "features.tsv")
df = pd.read_csv(path, header=None)
pd.read_csv(path, sep="\t", header=None).head()

In [ ]:
path = os.path.join(Nvec_mtx_directory, "barcodes.tsv")

df = pd.read_csv(path, header=None)

# split into two parts
split = df[0].str.split(" ", n=1, expand=True)

barcodes = split[1].str.replace('"', '', regex=False)

barcodes.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "barcodes.tsv")

# read tsv skipping first row
df = pd.read_csv(path, sep="\t", header=None, skiprows=1)

# overwrite original file
df.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "barcodes.tsv")
df = pd.read_csv(path, header=None)
pd.read_csv(path, header=None).head()

In [ ]:
path = os.path.join(Nvec_mtx_directory, "metadata.tsv")

df = pd.read_csv(path, header=None)

# split into two parts
split = df[0].str.split(" ", n=1, expand=True)

barcodes = split[1].str.replace('"', '', regex=False)

barcodes.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "metadata.tsv")

# read tsv skipping first row
df = pd.read_csv(path, sep="\t", header=None, skiprows=1)

# overwrite original file
df.to_csv(path, sep="\t", header=False, index=False)

In [ ]:
path = os.path.join(Nvec_mtx_directory, "metadata.tsv")
df = pd.read_csv(path, header=None)
pd.read_csv(path, header=None).head()

In [ ]:
matrix_path = os.path.join(Nvec_mtx_directory, "matrix.mtx")
features_path = os.path.join(Nvec_mtx_directory, "features.tsv")
barcodes_path = os.path.join(Nvec_mtx_directory, "barcodes.tsv")
metadata_path = os.path.join(Nvec_mtx_directory, "metadata.tsv")

# load matrix
adata = sc.read_mtx(matrix_path).T

# load features with correct encoding
genes = pd.read_csv(features_path, sep="\t", header=None, encoding="ascii")


# load barcodes with same encoding
barcodes = pd.read_csv(barcodes_path, header=None, encoding="ascii")


# assign names
adata.var_names = genes[1].astype(str)
adata.var["gene_ids"] = genes[0].values
adata.obs_names = barcodes[0].astype(str)

adata.var_names_make_unique()

#load cluster names with same encoding
clusters = pd.read_csv(metadata_path, header=None, encoding="ascii")
clusters.index = barcodes[0].astype(str)
adata.obs['cell_type'] = clusters.loc[adata.obs_names, 0]
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

# save
adata.write(output_h5ad_file)

print(adata)

In [ ]:
#once all the datasets are .h5ad files, combine them for each specie to make cnidoglandular files
adata1 = sc.read_h5ad("Nvec_cnido.h5ad")
adata2 = sc.read_h5ad("Nvec_neurogland.h5ad")
adata3 = sc.read_h5ad("Nvec_mucous.h5ad")

adata = sc.concat([adata1, adata2, adata3])

In [ ]:
print(adata.shape)        # cells × genes
print(adata.obs.head())   # metadata
print(adata.var.head()) 

In [ ]:
adata.obs["cell_type"] = (
    adata.obs["cell_type"]
    .astype(str)
    .astype("category")
)
adata.write("Nvec_cnido_neurogland_mucous_combined.h5ad")

In [ ]:
!pip install ipywidgets

In [ ]:
import samap
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder,
                            sankey_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM

In [ ]:
 #need this older version of scipy to deal with some matrix incomaptibility issues in the following steps
!pip install scipy==1.12.0

In [ ]:
Ap = sc.read_h5ad('Apoc_cnido_neurogland_mucous_combine.h5ad')
Nv = sc.read_h5ad('Nvec_cnido_neurogland_mucous_combine.h5ad')

sam1 = SAM(Ap)
sam1.run()

sam2 = SAM(Nv)
sam2.run()

sams = {'Ap': sam1, 'Nv': sam2}

In [ ]:
sm = SAMAP(
        sams,
        f_maps = './maps/',
    )

In [ ]:
sm.run()
samap = sm.samap 

In [12]:
keys = {'Ap':'cell_type', 'Nv':'cell_type'}

In [13]:
D,MappingTable = get_mapping_scores(sm, keys)

In [ ]:
D,MappingTable.to_csv("ApNv_cnido_neurogland_mucous_mapping_scores.csv")

In [ ]:
gpf = GenePairFinder(sm,keys=keys)

In [ ]:
gene_pairs = gpf.find_all(align_thr=0.2)

In [118]:
gene_pairs.to_csv("gene_pairs.csv")

In [ ]:
A=pd.read_csv('Apoc_EGG.tsv',sep='\t',index_col=0)
B=pd.read_csv('Nvec_EGG.tsv',sep='\t',index_col=0)

In [ ]:
EGGs={'Ap':A,'Nv':B}

In [ ]:
FE = FunctionalEnrichment(sm,EGGs, col_key='KOG', keys=keys, align_thr=0.2)

In [ ]:
ENRICHMENT_SCORES,NUM_ENRICHED_GENES,ENRICHED_GENES = FE.calculate_enrichment(verbose=True)

In [ ]:
FE.plot_enrichment()
import matplotlib.pyplot as plt
plt.tight_layout()
plt.savefig("samap_NvAp_CNGM_FE_label.pdf", bbox_inches="tight")